<table align="left">
  <tr>
 <td><img src="icecreamtruck.jpeg" alt="Ice Cream Truck" width="150"/></td>
 <td align="left"><h1>Introduction to Code Agents</h1></td>

In [1]:
# Warning control
import warnings

warnings.filterwarnings("ignore")

import os
import io
import IPython.display
from PIL import Image
import base64

from dotenv import load_dotenv, find_dotenv

_ = load_dotenv() # read local .env file

from huggingface_hub import login

login(os.environ['HF_API_KEY'])

In [2]:
import pandas as pd

suppliers_data = {
    "name": [
        "Montreal Ice Cream Co",
        "Brain Freeze Brothers",
        "Toronto Gelato Ltd",
        "Buffalo Scoops",
        "Vermont Creamery",
    ],
    "location": [
        "Montreal, QC",
        "Burlington, VT",
        "Toronto, ON",
        "Buffalo, NY",
        "Portland, ME",
    ],
    "distance_km": [120, 85, 400, 220, 280],
    "canadian": [True, False, True, False, False],
    "price_per_liter": [1.95, 1.91, 1.82, 2.43, 2.33],
    "tasting_fee": [0, 12.50, 30.14, 42.00, 0.20],
}

data_description = """Suppliers have an additional tasting fee: that is a fixed fee applied to each order to taste the ice cream."""
suppliers_df = pd.DataFrame(suppliers_data)
suppliers_df

,name,location,distance_km,canadian,price_per_liter,tasting_fee
0,Montreal Ice Cream Co,"Montreal, QC",120,True,1.95,0.00
1,Brain Freeze Brothers,"Burlington, VT",85,False,1.91,12.50
2,Toronto Gelato Ltd,"Toronto, ON",400,True,1.82,30.14
3,Buffalo Scoops,"Buffalo, NY",220,False,2.43,42.00
4,Vermont Creamery,"Portland, ME",280,False,2.33,0.20


## Setup tools

In [3]:
import numpy as np

def calculate_daily_supplier_price(row):
    order_volume = 30

    # Calculate raw product cost
    product_cost = row["price_per_liter"] * order_volume

    # Calculate transport cost
    trucks_needed = np.ceil(order_volume / 300)
    cost_per_km = 1.20
    transport_cost = row["distance_km"] * cost_per_km * trucks_needed

    # Calculate tariffs for ice cream imported from Canada
    tariff = product_cost * np.pi / 50 * row["canadian"]

    # Get total cost
    total_cost = product_cost + transport_cost + tariff + row["tasting_fee"]
    return total_cost

suppliers_df["daily_price"] = suppliers_df.apply(calculate_daily_supplier_price, axis=1)
display(suppliers_df)

# Let's remove this column now.
suppliers_df = suppliers_df.drop("daily_price", axis=1)

,name,location,distance_km,canadian,price_per_liter,tasting_fee,daily_price
0,Montreal Ice Cream Co,"Montreal, QC",120,True,1.95,0.00,206.175663
1,Brain Freeze Brothers,"Burlington, VT",85,False,1.91,12.50,171.800000
2,Toronto Gelato Ltd,"Toronto, ON",400,True,1.82,30.14,568.170619
3,Buffalo Scoops,"Buffalo, NY",220,False,2.43,42.00,378.900000
4,Vermont Creamery,"Portland, ME",280,False,2.33,0.20,406.100000


In [5]:
# First we make a few tools
from smolagents import tool

@tool
def calculate_transport_cost(distance_km: float, order_volume: float) -> float:
    """
    Calculate transportation cost based on distance and order size.
    Refrigerated transport costs $1.2 per kilometer and has a capacity of 300 liters.

    Args:
        distance_km: the distance in kilometers
        order_volume: the order volume in liters
    """
    trucks_needed = np.ceil(order_volume / 300)
    cost_per_km = 1.20
    return distance_km * cost_per_km * trucks_needed


@tool
def calculate_tariff(base_cost: float, is_canadian: bool) -> float:
    """
    Calculates tariff for Canadian imports. Returns the tariff only, not the total cost.
    Assumes tariff on dairy products from Canada is worth 2 * pi / 100, approx 6.2%

    Args:
        base_cost: the base cost of goods, not including transportation cost.
        is_canadian: wether the import is from Canada.
    """
    if is_canadian:
        return base_cost * np.pi / 50
    return 0

In [6]:
calculate_transport_cost.description

'Calculate transportation cost based on distance and order size.\nRefrigerated transport costs $1.2 per kilometer and has a capacity of 300 liters.'

## Setup the Model

In [19]:



from smolagents import CodeAgent, InferenceClientModel
from helper import get_huggingface_token

model = InferenceClientModel(
    model_id="Qwen/Qwen3.5-9B",
    temperature=0.1
)


## Setup the Code Agent

In [20]:
agent = CodeAgent(
    model=model,
    tools=[calculate_transport_cost, calculate_tariff],
    max_steps=10,
    additional_authorized_imports=["pandas", "numpy"],
    verbosity_level=2
)
agent.logger.console.width=66

In [21]:
agent.run(
    """Can you get me the transportation cost for 50 liters
    of ice cream over 10 kilometers?"""
)

╭─────────────────────────── New run ────────────────────────────╮
│                                                                │
│ Can you get me the transportation cost for 50 liters           │
│     of ice cream over 10 kilometers?                           │
│                                                                │
╰─ InferenceClientModel - Qwen/Qwen3.5-9B ───────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ───────────────────────────────────────
Thought: I need to calculate the transportation cost for 50 liters
of ice cream over 10 kilometers. I have access to the             
`calculate_transport_cost` function which takes distance_km and   
order_volume as parameters. I'll call this function with the given
values.                                                           
                                                                  
<code>                                                            
cost = calculate_transport_cost(distance_km=10, order_volume=50)  
print(f"The transportation cost is: ${cost}")                     

─ Executing parsed code: ─────────────────────────────────────── 
  cost = calculate_transport_cost(distance_km=10,                 
  order_volume=50)                                                
  print(f"The transportation cost is: ${cost}")                   
 ────────────────────────────────────────────────────────────────

Execution logs:
The transportation cost is: $12.0

Out: None

...

Output message of the LLM: ───────────────────────────────────────
Thought: The transportation cost has been calculated as $12.0. Now
I need to provide the final answer using the final_answer tool.   
                                                                  
<code>                                                            
final_answer(12.0)                                                

─ Executing parsed code: ─────────────────────────────────────── 
  final_answer(12.0)                                              
 ────────────────────────────────────────────────────────────────

Final answer: 12.0

[Step 2: Duration 3.48 seconds| Input tokens: 4,880 | Output 
tokens: 301]

12.0

## Give the agent detailed instructions

In [22]:
task = """Here is a dataframe of different ice cream suppliers.
Could you give me a comparative table (as a dataframe) of the total
daily price for getting daily ice cream delivery from each of them,
given that we need exactly 30 liters of ice cream per day? Take
into account transportation cost and tariffs.
"""
agent.logger.level = 1 # Lower verbosity level
agent.run(
    task,
    additional_args={"suppliers_data": suppliers_df, "data_description": data_description},
)

╭─────────────────────────── New run ────────────────────────────╮
│                                                                │
│ Here is a dataframe of different ice cream suppliers.          │
│ Could you give me a comparative table (as a dataframe) of the  │
│ total                                                          │
│ daily price for getting daily ice cream delivery from each of  │
│ them,                                                          │
│ given that we need exactly 30 liters of ice cream per day?     │
│ Take                                                           │
│ into account transportation cost and tariffs.                  │
│                                                                │
│ You have been provided with these additional arguments, that   │
│ you can access directly using the keys as variables:           │
│ {'suppliers_data':                     name        location    │
│ distance_km  canadian  \                                       │
│ 0  Montreal Ice Cream Co    Montreal, QC          120          │
│ True                                                           │
│ 1  Brain Freeze Brothers  Burlington, VT           85          │
│ False                                                          │
│ 2     Toronto Gelato Ltd     Toronto, ON          400          │
│ True                                                           │
│ 3         Buffalo Scoops     Buffalo, NY          220          │
│ False                                                          │
│ 4       Vermont Creamery    Portland, ME          280          │
│ False                                                          │
│                                                                │
│    price_per_liter  tasting_fee                                │
│ 0             1.95         0.00                                │
│ 1             1.91        12.50                                │
│ 2             1.82        30.14                                │
│ 3             2.43        42.00                                │
│ 4             2.33         0.20  , 'data_description':         │
│ 'Suppliers have an additional tasting fee: that is a fixed fee │
│ applied to each order to taste the ice cream.'}.               │
│                                                                │
╰─ InferenceClientModel - Qwen/Qwen3.5-9B ───────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ─────────────────────────────────────── 
  import pandas as pd                                             
                                                                  
  # The dataframe is already provided as suppliers_data           
  print("Suppliers data:")                                        
  print(suppliers_data)                                           
 ────────────────────────────────────────────────────────────────

Execution logs:
Suppliers data:
                    name        location  distance_km  canadian  \
0  Montreal Ice Cream Co    Montreal, QC          120      True   
1  Brain Freeze Brothers  Burlington, VT           85     False   
2     Toronto Gelato Ltd     Toronto, ON          400      True   
3         Buffalo Scoops     Buffalo, NY          220     False   
4       Vermont Creamery    Portland, ME          280     False   

   price_per_liter  tasting_fee  
0             1.95         0.00  
1             1.91        12.50  
2             1.82        30.14  
3             2.43        42.00  
4             2.33         0.20  

Out: None

[Step 1: Duration 5.41 seconds| Input tokens: 2,623 | Output 
tokens: 454]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ─────────────────────────────────────── 
  final_answer(result_df[['name', 'total_daily_price']])          
 ────────────────────────────────────────────────────────────────

Final answer:                     name  total_daily_price
0  Montreal Ice Cream Co         215.223450
1  Brain Freeze Brothers         171.800000
2     Toronto Gelato Ltd         598.329909
3         Buffalo Scoops         378.900000
4       Vermont Creamery         406.100000

[Step 4: Duration 1.72 seconds| Input tokens: 14,271 | Output 
tokens: 1,306]

,name,total_daily_price
0,Montreal Ice Cream Co,215.223450
1,Brain Freeze Brothers,171.800000
2,Toronto Gelato Ltd,598.329909
3,Buffalo Scoops,378.900000
4,Vermont Creamery,406.100000


## Compare to traditional tool calling.

In [24]:
from smolagents import ToolCallingAgent

model = InferenceClientModel(
    "Qwen/Qwen3.5-9B",
    #"Qwen/Qwen2.5-72B-Instruct", ### model no longer available in provider together
    provider="together", # Choose a specific inference provider
    temperature=0.6
)
agent = ToolCallingAgent(
    model=model,
    tools=[calculate_transport_cost, calculate_tariff],
    max_steps=20,
)
agent.logger.console.width=66

<div style="font-size: 14px; background-color: #fff9b0; padding: 12px; border-left: 4px solid #facc15;">
  <strong>Note:</strong> During this run, the agent may generate code that creates an error. This is OK. The agent will try to fix the error.
</div>


In [25]:
output = agent.run(
    task,
    additional_args={"suppliers_data": suppliers_df, "data_description": data_description},
)
print(output)

╭─────────────────────────── New run ────────────────────────────╮
│                                                                │
│ Here is a dataframe of different ice cream suppliers.          │
│ Could you give me a comparative table (as a dataframe) of the  │
│ total                                                          │
│ daily price for getting daily ice cream delivery from each of  │
│ them,                                                          │
│ given that we need exactly 30 liters of ice cream per day?     │
│ Take                                                           │
│ into account transportation cost and tariffs.                  │
│                                                                │
│ You have been provided with these additional arguments, that   │
│ you can access directly using the keys as variables:           │
│ {'suppliers_data':                     name        location    │
│ distance_km  canadian  \                                       │
│ 0  Montreal Ice Cream Co    Montreal, QC          120          │
│ True                                                           │
│ 1  Brain Freeze Brothers  Burlington, VT           85          │
│ False                                                          │
│ 2     Toronto Gelato Ltd     Toronto, ON          400          │
│ True                                                           │
│ 3         Buffalo Scoops     Buffalo, NY          220          │
│ False                                                          │
│ 4       Vermont Creamery    Portland, ME          280          │
│ False                                                          │
│                                                                │
│    price_per_liter  tasting_fee                                │
│ 0             1.95         0.00                                │
│ 1             1.91        12.50                                │
│ 2             1.82        30.14                                │
│ 3             2.43        42.00                                │
│ 4             2.33         0.20  , 'data_description':         │
│ 'Suppliers have an additional tasting fee: that is a fixed fee │
│ applied to each order to taste the ice cream.'}.               │
│                                                                │
╰─ InferenceClientModel - Qwen/Qwen3.5-9B ───────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭────────────────────────────────────────────────────────────────╮
│ Calling tool: 'calculate_transport_cost' with arguments:       │
│ {'distance_km': 120, 'order_volume': 30}                       │
╰────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────────────────╮
│ Calling tool: 'calculate_transport_cost' with arguments:       │
│ {'distance_km': 85, 'order_volume': 30}                        │
╰────────────────────────────────────────────────────────────────╯

Observations: 144.0

...

[Step 2: Duration 67.96 seconds| Input tokens: 4,302 | Output 
tokens: 2,017]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error while generating output:
402 Client Error: Payment Required for url: 
https://router.huggingface.co/together/v1/chat/completions 
(Request ID: 
Root=1-6a5f1bc2-6dc9da28109a84b32e9545ea;aeb6b2b1-9018-4c27-ba3f-5
20ab8403dd8)

You have depleted your monthly included credits. Purchase pre-paid
credits to continue using Inference Providers. Alternatively, 
subscribe to PRO to get 20x more included usage.

[Step 3: Duration 0.08 seconds]

AgentGenerationError: Error while generating output:
402 Client Error: Payment Required for url: https://router.huggingface.co/together/v1/chat/completions (Request ID: Root=1-6a5f1bc2-6dc9da28109a84b32e9545ea;aeb6b2b1-9018-4c27-ba3f-520ab8403dd8)

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.